In [0]:
# Create table with NOT NULL on custid (required for primary key)
spark.sql("""
    CREATE OR REPLACE TABLE dev.test1_bronze.customers (
        custid LONG NOT NULL,
        fname STRING,
        lname STRING,
        SSN LONG,
        email STRING,
        CONSTRAINT customers_pk PRIMARY KEY (custid)
    )
""")

# Insert 20 sample records
spark.sql("""
    INSERT INTO dev.test1_bronze.customers VALUES
    (10001, 'John', 'Smith', 123456789, 'john.smith@email.com'),
    (10002, 'Jane', 'Doe', 234567890, 'jane.doe@email.com'),
    (10003, 'Michael', 'Johnson', 345678901, 'michael.johnson@email.com'),
    (10004, 'Emily', 'Williams', 456789012, 'emily.williams@email.com'),
    (10005, 'David', 'Brown', 567890123, 'david.brown@email.com'),
    (10006, 'Sarah', 'Jones', 678901234, 'sarah.jones@email.com'),
    (10007, 'Robert', 'Garcia', 789012345, 'robert.garcia@email.com'),
    (10008, 'Lisa', 'Martinez', 890123456, 'lisa.martinez@email.com'),
    (10009, 'James', 'Davis', 901234567, 'james.davis@email.com'),
    (10010, 'Jennifer', 'Rodriguez', 112345678, 'jennifer.rodriguez@email.com'),
    (10011, 'William', 'Wilson', 223456789, 'william.wilson@email.com'),
    (10012, 'Amanda', 'Anderson', 334567890, 'amanda.anderson@email.com'),
    (10013, 'Daniel', 'Thomas', 445678901, 'daniel.thomas@email.com'),
    (10014, 'Jessica', 'Taylor', 556789012, 'jessica.taylor@email.com'),
    (10015, 'Christopher', 'Moore', 667890123, 'christopher.moore@email.com'),
    (10016, 'Ashley', 'Jackson', 778901234, 'ashley.jackson@email.com'),
    (10017, 'Matthew', 'Martin', 889012345, 'matthew.martin@email.com'),
    (10018, 'Stephanie', 'Lee', 990123456, 'stephanie.lee@email.com'),
    (10019, 'Andrew', 'Harris', 101234567, 'andrew.harris@email.com'),
    (10020, 'Nicole', 'Clark', 212345678, 'nicole.clark@email.com')
""")

display(spark.table("dev.test1_bronze.customers"))

custid,fname,lname,SSN,email
10001,John,Smith,123456789,john.smith@email.com
10002,Jane,Doe,234567890,jane.doe@email.com
10003,Michael,Johnson,345678901,michael.johnson@email.com
10004,Emily,Williams,456789012,emily.williams@email.com
10005,David,Brown,567890123,david.brown@email.com
10006,Sarah,Jones,678901234,sarah.jones@email.com
10007,Robert,Garcia,789012345,robert.garcia@email.com
10008,Lisa,Martinez,890123456,lisa.martinez@email.com
10009,James,Davis,901234567,james.davis@email.com
10010,Jennifer,Rodriguez,112345678,jennifer.rodriguez@email.com


In [0]:
# dbt-style incremental model: merge bronze into silver

# Create silver table if it doesn't exist
spark.sql("""
    CREATE TABLE IF NOT EXISTS dev.test1_silver.cust_silv (
        custid LONG NOT NULL,
        fname STRING,
        lname STRING,
        SSN LONG,
        email STRING,
        CONSTRAINT cust_silv_pk PRIMARY KEY (custid)
    )
""")

# MERGE: insert new records, update existing ones (incremental strategy)
spark.sql("""
    MERGE INTO dev.test1_silver.cust_silv AS target
    USING dev.test1_bronze.customers AS source
    ON target.custid = source.custid
    WHEN MATCHED THEN
        UPDATE SET
            target.fname = source.fname,
            target.lname = source.lname,
            target.SSN = source.SSN,
            target.email = source.email
    WHEN NOT MATCHED THEN
        INSERT (custid, fname, lname, SSN, email)
        VALUES (source.custid, source.fname, source.lname, source.SSN, source.email)
""")

display(spark.table("dev.test1_silver.cust_silv"))

custid,fname,lname,SSN,email
10001,John,Smith,123456789,john.smith@email.com
10002,Jane,Doe,234567890,jane.doe@email.com
10003,Michael,Johnson,345678901,michael.johnson@email.com
10004,Emily,Williams,456789012,emily.williams@email.com
10005,David,Brown,567890123,david.brown@email.com
10006,Sarah,Jones,678901234,sarah.jones@email.com
10007,Robert,Garcia,789012345,robert.garcia@email.com
10008,Lisa,Martinez,890123456,lisa.martinez@email.com
10009,James,Davis,901234567,james.davis@email.com
10010,Jennifer,Rodriguez,112345678,jennifer.rodriguez@email.com


In [0]:
# Scenario: Lisa's email changed, but source system only allows inserts.
# So it arrives as a new record with custid 10021 in the bronze table.

spark.sql("""
    INSERT INTO dev.test1_bronze.customers VALUES
    (10021, 'Lisa', 'Martinez', 890123456, 'lisa.martinez1@email.com')
""")

display(spark.table("dev.test1_bronze.customers").filter("fname = 'Lisa'"))

custid,fname,lname,SSN,email
10008,Lisa,Martinez,890123456,lisa.martinez@email.com
10021,Lisa,Martinez,890123456,lisa.martinez1@email.com


In [0]:
# Scenario: Lisa's email changed again to lisa.martinez3@email.com
# Source system only allows inserts, so it arrives as custid 10022 in bronze.
# (Previously deleted in Cell 6, so custid 10022 is available again)

spark.sql("""
    INSERT INTO dev.test1_bronze.customers VALUES
    (10022, 'Lisa', 'Martinez', 890123456, 'lisa.martinez3@email.com')
""")

display(spark.table("dev.test1_bronze.customers").filter("fname = 'Lisa'"))

custid,fname,lname,SSN,email
10008,Lisa,Martinez,890123456,lisa.martinez@email.com
10021,Lisa,Martinez,890123456,lisa.martinez1@email.com
10022,Lisa,Martinez,890123456,lisa.martinez3@email.com


In [0]:
# SCD Type 1: Keep only the latest version of each customer in silver.
# Since bronze has multiple records for the same person (10008, 10021, 10022 for Lisa),
# we deduplicate by SSN (natural key), keeping the highest custid as the most recent.

# Step 1: Deduplicate bronze - pick the latest record per SSN (natural key)

###

#Here's the SCD Type 1 approach:
#Deduplicates bronze — partitions by SSN (natural key) and keeps only the record with the highest custid (most recent insert). So for Lisa, it picks custid 10022 with lisa.martinez3@email.com.
#Merges into silver on SSN — when matched, it overwrites all fields (including custid and email) with the latest values. No history is preserved (Type 1 behavior).
#This ensures Lisa appears only once in silver with her most current email. 

####
spark.sql("""
    CREATE OR REPLACE TEMP VIEW bronze_deduped AS
    SELECT custid, fname, lname, SSN, email
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY SSN ORDER BY custid DESC) AS rn
        FROM dev.test1_bronze.customers
    )
    WHERE rn = 1
""")

# Step 2: SCD Type 1 MERGE into silver using SSN as the natural key
# When matched: overwrite all fields with the latest values (Type 1 = no history)
# When not matched: insert new customer
spark.sql("""
    MERGE INTO dev.test1_silver.cust_silv AS target
    USING bronze_deduped AS source
    ON target.SSN = source.SSN
    WHEN MATCHED THEN
        UPDATE SET
            target.custid = source.custid,
            target.fname = source.fname,
            target.lname = source.lname,
            target.email = source.email
    WHEN NOT MATCHED THEN
        INSERT (custid, fname, lname, SSN, email)
        VALUES (source.custid, source.fname, source.lname, source.SSN, source.email)
""")

print("--- Silver table after SCD Type 1 merge ---")
display(spark.table("dev.test1_silver.cust_silv").orderBy("custid"))

--- Silver table after SCD Type 1 merge ---


custid,fname,lname,SSN,email
10001,John,Smith,123456789,john.smith@email.com
10002,Jane,Doe,234567890,jane.doe@email.com
10003,Michael,Johnson,345678901,michael.johnson@email.com
10004,Emily,Williams,456789012,emily.williams@email.com
10005,David,Brown,567890123,david.brown@email.com
10006,Sarah,Jones,678901234,sarah.jones@email.com
10007,Robert,Garcia,789012345,robert.garcia@email.com
10009,James,Davis,901234567,james.davis@email.com
10010,Jennifer,Rodriguez,112345678,jennifer.rodriguez@email.com
10011,William,Wilson,223456789,william.wilson@email.com
